# TabPack

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yandex-research/tabpack/blob/main/examples/tabpack_sklearn.ipynb)

This notebook shows how to use `tabpackage.sklearn` to train TabPack on a classification task.

In [ ]:
%pip install tabpackage[sklearn]

In [ ]:
# @title Setup and imports
import os

# Make the code runnable on Apple Silicon for PyTorch <2.12.
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

from tabpackage.sklearn import TabPackClassifier

seed = 0

## Data

This example uses pandas. The interface accepts:

| Features (`X`) | Numerical columns | Categorical columns |
| --- | --- | --- |
| pandas DataFrame | `float32` | String dtype, or `category` with string labels |
| Polars DataFrame | `Float32` | `String`, `Categorical`, or `Enum` |
| NumPy array | `float32` | Not supported |

For classification, `y` is a NumPy array or pandas/Polars Series with an
`int64` or string dtype.

Categorical values are automatically ordinal-encoded, and unseen categories in
test data are supported. Keep column names, order, and dtypes consistent across
splits.

### Preprocessing

Missing values are not imputed; handle them before passing data to the estimator.
Numerical features must be finite. Below, we apply a quantile transform explicitly.

### Validation data

TabPack needs validation data not only for early stopping, but also for building
the ensemble used for predictions:

- For the first `fit`, an additional split is **required**. The typical baseline
  is to use the same validation data for early stopping and ensemble construction.
- Exclude this split from fitting preprocessing parameters (e.g. feature
  statistics), and respect dataset-specific splitting rules (e.g. time-based
  splitting).
- TabPack may struggle on small datasets, where the validation split may not be
  representative enough for reliable early stopping and ensemble construction.

After the first `fit`, `refit` can use the union of training and validation data.

In [15]:
# Numerical features and labels.
X_num, y_array = make_classification(
    n_samples=20_000,
    n_features=8,
    n_classes=3,
    n_informative=3,
    n_redundant=2,
    random_state=seed,
)
numerical_columns = [f'num_{i}' for i in range(X_num.shape[1])]
X = pd.DataFrame(X_num, columns=numerical_columns, dtype='float32')
y = pd.Series(y_array, name='target', dtype='int64')

# Categorical features are encoded automatically from string columns.
rng = np.random.default_rng(seed)
for i, cardinality in enumerate([4, 7]):
    X[f'cat_{i}'] = pd.Series(
        rng.choice([f'value_{j}' for j in range(cardinality)], size=len(X)),
        dtype='string',
    )

# Split into training, validation, and test data.
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=0.2, stratify=y, random_state=seed
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.2, stratify=y.iloc[train_idx], random_state=seed + 1
)
# Test data may contain categorical values not seen during training.
X.loc[test_idx[:5], 'cat_0'] = 'unseen'

split = {'train': train_idx, 'val': val_idx, 'test': test_idx}
raw_datasets = {
    name: {'X': X.iloc[idx], 'y': y.iloc[idx]} for name, idx in split.items()
}

print('Training data')
display(raw_datasets['train']['X'].head())
print('Test data')
display(raw_datasets['test']['X'].head())

Training data


,num_0,num_1,num_2,num_3,num_4,num_5,num_6,num_7,cat_0,cat_1
16877,-0.254253,-0.201380,2.053693,1.611623,-0.310795,0.267808,0.950319,0.743403,value_3,value_0
15548,-0.124877,0.818227,0.746556,1.037497,-0.746239,2.526568,0.246961,-0.482379,value_3,value_3
18110,0.456350,-0.841465,-1.071347,-1.149274,1.776408,0.466469,0.350668,0.785449,value_2,value_1
7659,0.935767,3.237482,-3.456160,-0.534910,0.461310,0.052730,-0.281176,-2.904902,value_3,value_1
4064,-0.298531,-1.708302,-1.053829,0.799577,-0.165224,0.324307,-0.347831,0.507888,value_1,value_1


Test data


,num_0,num_1,num_2,num_3,num_4,num_5,num_6,num_7,cat_0,cat_1
15817,-0.211729,-0.031673,1.343233,0.681905,-0.410425,-0.746914,-0.573574,0.347712,unseen,value_0
2294,-0.245115,-0.620085,3.075859,-0.581141,0.164825,0.154965,-0.135466,1.542794,unseen,value_3
10174,0.338739,2.580980,-2.129117,-2.166498,-0.837068,1.616135,0.711256,-2.564619,unseen,value_6
13099,0.120782,1.175882,1.720985,1.821659,0.030813,0.191349,-0.615257,-0.017170,unseen,value_2
16173,0.375820,-0.213964,1.140476,-1.571228,1.571367,-1.039307,-1.462003,1.162969,unseen,value_0


In [16]:
def preprocess_datasets(raw_datasets):
    datasets = {
        name: {'X': dataset['X'].copy(), 'y': dataset['y']}
        for name, dataset in raw_datasets.items()
    }
    X_num_train = datasets['train']['X'][numerical_columns]
    num_transformer = QuantileTransformer(
        n_quantiles=max(min(len(X_num_train) // 30, 1000), 10),
        output_distribution='normal',
        subsample=10**9,
        random_state=seed,
    )
    # Break ties between equal feature values when fitting the transformer.
    noise = (
        np.random.default_rng(seed)
        .normal(0.0, 1e-5, X_num_train.shape)
        .astype(np.float32)
    )
    num_transformer.fit(X_num_train + noise)
    for dataset in datasets.values():
        dataset['X'][numerical_columns] = num_transformer.transform(
            dataset['X'][numerical_columns]
        ).astype(np.float32)
    return datasets


datasets = preprocess_datasets(raw_datasets)

## Fit and predict

In [14]:
model = TabPackClassifier(random_state=seed)
model.fit(
    datasets['train']['X'],
    datasets['train']['y'],
    extra_set=(datasets['val']['X'], datasets['val']['y']),
);

                                                                               
         training          |        models        |         best       
    time | epoch |   loss  |  finished | running  |    model | ensemble
-----------------------------------------------------------------------
00:00:09 |     1 |  1.066  |         0 |      64  |   0.8447 |   0.8478
00:00:19 |     2 |  0.894  |         0 |      64  |   0.8522 |   0.8578        
00:00:29 |     3 |  0.774  |         0 |      64  |   0.8547 |   0.8597        
00:00:39 |     4 |  0.677  |         0 |      64  |   0.8587 |   0.8628        
00:00:48 |     5 |  0.603  |         0 |      64  |   0.8587 |   0.8628        
00:00:58 |     6 |  0.546  |         0 |      64  |   0.8597 |   0.8647        
00:01:08 |     7 |  0.497  |         0 |      64  |   0.8612 |   0.8647        
00:01:17 |     8 |  0.458  |         0 |      64  |   0.8612 |   0.8653        
00:01:27 |     9 |  0.426  |         0 |      64  |   0.8612 |   0.8653 

In [17]:
y_pred = model.predict(datasets['test']['X'])
y_proba = model.predict_proba(datasets['test']['X'])

## Refit on full training data

Retrain the selected ensemble members from scratch on training + validation
data, reusing their hyperparameters and training durations. No additional
validation split is needed. This may improve performance on some datasets.

In [18]:
# Merge the training and validation data.
retraining_raw_datasets = {
    'train': {
        k: pd.concat([v, raw_datasets['val'][k]])
        for k, v in raw_datasets['train'].items()
    },
    'test': raw_datasets['test'],
}
# Redo preprocessing on full training data, including transforming the test data.
retraining_datasets = preprocess_datasets(retraining_raw_datasets)

model.refit(retraining_datasets['train']['X'], retraining_datasets['train']['y']);

                                                                               
         training          |        models      
    time | epoch |   loss  |  finished | running
------------------------------------------------
00:00:01 |     1 |  0.997  |         0 |       7
00:00:02 |     2 |  0.654  |         0 |       7                               
00:00:04 |     3 |  0.590  |         1 |       6                               
00:00:05 |     4 |  0.572  |         1 |       6                               
00:00:07 |     5 |  0.501  |         2 |       5                               
00:00:08 |     6 |  0.465  |         2 |       5                               
00:00:09 |     7 |  0.414  |         3 |       4                               
00:00:10 |     8 |  0.393  |         4 |       3                               
00:00:11 |     9 |  0.422  |         4 |       3                               
00:00:11 |    10 |  0.405  |         4 |       3                               
00:0

## Results

In [20]:
y_pred_refit = model.predict(retraining_datasets['test']['X'])
y_proba_refit = model.predict_proba(retraining_datasets['test']['X'])
y_test = raw_datasets['test']['y']

results = pd.DataFrame(
    {
        'Test accuracy': [
            accuracy_score(y_test, y_pred),
            accuracy_score(y_test, y_pred_refit),
        ],
    },
    index=['Fit: train only', 'Refit: train + validation'],
)
results.round(4)

,Test accuracy
Fit: train only,0.8545
Refit: train + validation,0.8558


## PyTorch Interface

See [the PyTorch example](./tabpack.py) which demonstates the tabpackage PyTorch API, a more hackable interface. The example explains more configuration options, efficiency settings, and exporting
predictors with `torch.save`, ONNX, and `torch.export`.